# Análise de dados TCP-CII

In [9]:
import pandas as pd

In [10]:
df = pd.read_csv('./T CELL/DENV 4 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNQTWQIEKASLIEV,206,220,15,HLA-DRB1*01:01,42,0.06,WQIEKASLI,0.974156,0.06
1,1,QYKFQPESPARLASA,31,45,15,HLA-DRB1*01:01,7,0.20,FQPESPARL,0.952735,0.20
2,1,LTVVAGDVKGVLTKG,86,100,15,HLA-DRB1*03:01,18,0.43,VAGDVKGVL,0.882731,0.43
3,1,GSGIFVVDNVHTWTE,16,30,15,HLA-DRB3*01:01,4,0.45,FVVDNVHTW,0.676660,0.45
4,1,ERRAWNSLEVEDYGF,146,160,15,HLA-DQA1*05:01/DQB1*02:01,30,0.49,WNSLEVEDY,0.652001,0.49
...,...,...,...,...,...,...,...,...,...,...,...
1831,1,KLVTQWCCRSCTMPP,306,320,15,HLA-DRB1*04:01,62,100.00,LVTQWCCRS,0.000064,100.00
1832,1,WCCRSCTMPPLRFLG,311,325,15,HLA-DRB3*01:01,63,100.00,CRSCTMPPL,0.000056,100.00
1833,1,WCCRSCTMPPLRFLG,311,325,15,HLA-DRB1*04:05,63,100.00,SCTMPPLRF,0.000054,100.00
1834,1,WCCRSCTMPPLRFLG,311,325,15,HLA-DRB1*04:01,63,100.00,RSCTMPPLR,0.000037,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [11]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNQTWQIEKASLIEV,206,220,15,HLA-DRB1*01:01,42,0.06,WQIEKASLI,0.974156,0.06
1,1,QYKFQPESPARLASA,31,45,15,HLA-DRB1*01:01,7,0.20,FQPESPARL,0.952735,0.20
2,1,LTVVAGDVKGVLTKG,86,100,15,HLA-DRB1*03:01,18,0.43,VAGDVKGVL,0.882731,0.43
3,1,GSGIFVVDNVHTWTE,16,30,15,HLA-DRB3*01:01,4,0.45,FVVDNVHTW,0.676660,0.45
4,1,ERRAWNSLEVEDYGF,146,160,15,HLA-DQA1*05:01/DQB1*02:01,30,0.49,WNSLEVEDY,0.652001,0.49
...,...,...,...,...,...,...,...,...,...,...,...
57,1,TRLENVMWKQITNEL,61,75,15,HLA-DPA1*01:03/DPB1*02:01,13,4.50,LENVMWKQI,0.166688,4.50
58,1,YRQGYATQTVGPWHL,256,270,15,HLA-DQA1*05:01/DQB1*03:01,52,4.70,YATQTVGPW,0.330795,4.70
59,1,ITNELNYVLWEGGHD,71,85,15,HLA-DPA1*03:01/DPB1*04:02,15,4.70,LNYVLWEGG,0.137848,4.70
60,1,VVDNVHTWTEQYKFQ,21,35,15,HLA-DQA1*01:01/DQB1*05:01,5,4.70,VHTWTEQYK,0.014117,4.70


## Agrupando por pepitideos e agregando colunas pertinentes.

In [12]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAVHADMGY,186,200,2,1.700,"HLA-DRB1*01:01, HLA-DRB4*01:01"
1,ADMGYWIESSKNQTW,196,210,1,2.400,HLA-DRB1*04:01
2,AKIFTPEARNSTFLI,121,135,2,3.250,"HLA-DRB1*08:02, HLA-DRB1*11:01"
3,DFGECPGTTVTIQED,276,290,1,4.200,HLA-DQA1*03:01/DQB1*03:02
4,ERRAWNSLEVEDYGF,146,160,4,2.250,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
5,FSQHNYRQGYATQTV,251,265,1,3.100,HLA-DQA1*05:01/DQB1*03:01
6,GMFTTNIWMKFREGS,161,175,1,3.600,HLA-DPA1*01:03/DPB1*04:01
7,GPWHLGKLEIDFGEC,266,280,1,3.200,HLA-DRB4*01:01
8,GSGIFVVDNVHTWTE,16,30,4,2.200,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB1*04:01, HLA..."
9,HRLMSAAIKDQKAVH,181,195,1,1.600,HLA-DQA1*01:02/DQB1*06:02


# Filtragem por epítopos que presentes em mais de 2 alelos.

In [13]:
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= 2
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAVHADMGY,186,200,2,1.700,"HLA-DRB1*01:01, HLA-DRB4*01:01"
1,AKIFTPEARNSTFLI,121,135,2,3.250,"HLA-DRB1*08:02, HLA-DRB1*11:01"
2,ERRAWNSLEVEDYGF,146,160,4,2.250,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
3,GSGIFVVDNVHTWTE,16,30,4,2.200,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB1*04:01, HLA..."
4,ITNELNYVLWEGGHD,71,85,3,3.400,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
5,KNQTWQIEKASLIEV,206,220,12,2.950,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
6,LKYSWKTWGKAKIFT,111,125,2,2.150,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02"
7,QKAVHADMGYWIESS,191,205,2,2.745,"HLA-DRB1*03:01, HLA-DRB3*01:01"
8,QYKFQPESPARLASA,31,45,11,1.700,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
9,YGMEIRPLSEKEENM,331,345,2,2.750,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB1*12:01"


In [14]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,YRQGYATQTVGPWHL,256,270,3,1.400,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
1,QYKFQPESPARLASA,31,45,11,1.700,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
2,AAIKDQKAVHADMGY,186,200,2,1.700,"HLA-DRB1*01:01, HLA-DRB4*01:01"
3,LKYSWKTWGKAKIFT,111,125,2,2.150,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02"
4,GSGIFVVDNVHTWTE,16,30,4,2.200,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB1*04:01, HLA..."
5,ERRAWNSLEVEDYGF,146,160,4,2.250,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
6,QKAVHADMGYWIESS,191,205,2,2.745,"HLA-DRB1*03:01, HLA-DRB3*01:01"
7,YGMEIRPLSEKEENM,331,345,2,2.750,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB1*12:01"
8,KNQTWQIEKASLIEV,206,220,12,2.950,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
9,AKIFTPEARNSTFLI,121,135,2,3.250,"HLA-DRB1*08:02, HLA-DRB1*11:01"


In [15]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,YRQGYATQTVGPWHL,256,270,3,1.400,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
1,QYKFQPESPARLASA,31,45,11,1.700,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
2,AAIKDQKAVHADMGY,186,200,2,1.700,"HLA-DRB1*01:01, HLA-DRB4*01:01"
3,LKYSWKTWGKAKIFT,111,125,2,2.150,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02"
4,GSGIFVVDNVHTWTE,16,30,4,2.200,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB1*04:01, HLA..."
5,ERRAWNSLEVEDYGF,146,160,4,2.250,"HLA-DPA1*02:01/DPB1*01:01, HLA-DQA1*01:01/DQB1..."
6,QKAVHADMGYWIESS,191,205,2,2.745,"HLA-DRB1*03:01, HLA-DRB3*01:01"
7,YGMEIRPLSEKEENM,331,345,2,2.750,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB1*12:01"
8,KNQTWQIEKASLIEV,206,220,12,2.950,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
9,AKIFTPEARNSTFLI,121,135,2,3.250,"HLA-DRB1*08:02, HLA-DRB1*11:01"


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [16]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0     YRQGYATQTVGPWHL
1     QYKFQPESPARLASA
2     AAIKDQKAVHADMGY
3     LKYSWKTWGKAKIFT
4     GSGIFVVDNVHTWTE
5     ERRAWNSLEVEDYGF
6     QKAVHADMGYWIESS
7     YGMEIRPLSEKEENM
8     KNQTWQIEKASLIEV
9     AKIFTPEARNSTFLI
10    ITNELNYVLWEGGHD
Name: peptide, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [21]:
!seqkit grep -s -v -r -p '[-*X?]' 'denv4_NS1_final.fasta' > DENV4_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [23]:
conservacy_result = pd.read_csv('./ConservancyResult.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,NaN
1,2,NP 2,QYKFQPESPARLASA,15,98.66% (147/149),73.33%,100.00%,NaN
2,3,NP 3,AAIKDQKAVHADMGY,15,95.30% (142/149),86.67%,100.00%,NaN
3,4,NP 4,LKYSWKTWGKAKIFT,15,95.30% (142/149),93.33%,100.00%,NaN
4,5,NP 5,GSGIFVVDNVHTWTE,15,73.15% (109/149),86.67%,100.00%,NaN
5,6,NP 6,ERRAWNSLEVEDYGF,15,29.53% (44/149),80.00%,100.00%,NaN
6,7,NP 7,QKAVHADMGYWIESS,15,95.30% (142/149),86.67%,100.00%,NaN
7,8,NP 8,YGMEIRPLSEKEENM,15,83.22% (124/149),80.00%,100.00%,NaN
8,9,NP 9,KNQTWQIEKASLIEV,15,62.42% (93/149),80.00%,100.00%,NaN
9,10,NP 10,AKIFTPEARNSTFLI,15,71.81% (107/149),80.00%,100.00%,NaN


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [24]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details,percent_match
0,1,NP 1,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,NaN,62.42
1,2,NP 2,QYKFQPESPARLASA,15,98.66% (147/149),73.33%,100.00%,NaN,98.66
2,3,NP 3,AAIKDQKAVHADMGY,15,95.30% (142/149),86.67%,100.00%,NaN,95.30
3,4,NP 4,LKYSWKTWGKAKIFT,15,95.30% (142/149),93.33%,100.00%,NaN,95.30
4,5,NP 5,GSGIFVVDNVHTWTE,15,73.15% (109/149),86.67%,100.00%,NaN,73.15
5,6,NP 6,ERRAWNSLEVEDYGF,15,29.53% (44/149),80.00%,100.00%,NaN,29.53
6,7,NP 7,QKAVHADMGYWIESS,15,95.30% (142/149),86.67%,100.00%,NaN,95.30
7,8,NP 8,YGMEIRPLSEKEENM,15,83.22% (124/149),80.00%,100.00%,NaN,83.22
8,9,NP 9,KNQTWQIEKASLIEV,15,62.42% (93/149),80.00%,100.00%,NaN,62.42
9,10,NP 10,AKIFTPEARNSTFLI,15,71.81% (107/149),80.00%,100.00%,NaN,71.81


### Sort por percent_match e filtragem por percent_match >= 95.00

In [25]:
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= 95.0]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    )

conservacy_result_filtered

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details,percent_match
1,2,NP 2,QYKFQPESPARLASA,15,98.66% (147/149),73.33%,100.00%,NaN,98.66
10,11,NP 11,ITNELNYVLWEGGHD,15,97.32% (145/149),86.67%,100.00%,NaN,97.32
2,3,NP 3,AAIKDQKAVHADMGY,15,95.30% (142/149),86.67%,100.00%,NaN,95.30
3,4,NP 4,LKYSWKTWGKAKIFT,15,95.30% (142/149),93.33%,100.00%,NaN,95.30
6,7,NP 7,QKAVHADMGYWIESS,15,95.30% (142/149),86.67%,100.00%,NaN,95.30
